# TB-Trust — 04: Full evaluation and results (Phase 4)

Aggregates every fold into the cross-site comparison and prints the numbers the paper's tables are built from.

In [ ]:
# --- configuration ---------------------------------------------------------
# Defaults are the Kaggle paths. Every path is read from the environment first,
# so the same notebook runs unmodified on Kaggle, locally, or in CI -- which is
# also what lets these notebooks be executed as a test rather than only read.
import os

REPO = os.environ.get("TBTRUST_REPO", "/kaggle/working/tb-trust")
DATA = os.environ.get("TBTRUST_DATA", "/kaggle/input/tuberculosis-tb-chest-xray-dataset")
WORK = os.environ.get("TBTRUST_WORK", "/kaggle/working")
REPO_URL = os.environ.get("TBTRUST_REPO_URL", "https://github.com/AIscend-Research/tb-trust.git")

MANIFEST = f"{WORK}/manifest.csv"
OUT = f"{WORK}/outputs"
os.makedirs(WORK, exist_ok=True)
print("REPO:", REPO, "\nDATA:", DATA, "\nWORK:", WORK)

In [ ]:
# Enter the repo and make it importable. The install is skipped when the package
# already resolves, so re-running a notebook is cheap.
import importlib.util
import os
import subprocess
import sys

os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
if importlib.util.find_spec("tbtrust") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    importlib.invalidate_caches()
print("tbtrust ready from", REPO)

## 1. In-distribution reference

Written by notebook 02 from the **evaluated** random-split run, not from a training-time validation score.

In [ ]:
import json
from pathlib import Path

ref_path = Path(f"{OUT}/reference/montgomery/metrics.json")
assert ref_path.exists(), "run notebook 02 first -- the reference run has not been evaluated"
ref = json.loads(ref_path.read_text())
assert ref["split_mode"] == "random", (
    "the reference must come from a random-split run; a LOCO fold's accuracy is "
    "the deployment number, not the in-distribution baseline"
)
REFERENCE_ACCURACY = ref["robustness_sweep"]["0.5"]["accuracy"]
print("in-distribution reference accuracy:", round(REFERENCE_ACCURACY, 4))

## 2. Aggregate the leave-one-clinic-out sweep

In [ ]:
import subprocess
import sys

cmd = [sys.executable, "scripts/run_experiments.py",
       "--configs", "configs/loco_montgomery.yaml,configs/loco_shenzhen.yaml",
       "--checkpoints", f"{OUT}/baseline/montgomery/best.ckpt,{OUT}/baseline/shenzhen/best.ckpt",
       "--reference-accuracy", str(REFERENCE_ACCURACY),
       "--out-dir", f"{WORK}/loco_sweep",
       f"data.manifest={MANIFEST}"]
print("$", " ".join(cmd))
r = subprocess.run(cmd, capture_output=True, text=True)
print((r.stdout or "")[-3000:])
if r.returncode != 0:
    print((r.stderr or "")[-3000:])
    raise RuntimeError("run_experiments failed")

In [ ]:
report = json.loads(Path(f"{WORK}/loco_sweep/loco_sweep_report.json").read_text())
print(json.dumps({k: v for k, v in report.items() if k != "per_clinic"}, indent=2)[:3000])

## 3. Per-fold safe deferral, calibration, and conformal coverage

The uncertainty methods all score the *same* temperature-scaled probabilities and differ only in the signal the deferral policy ranks on — so AURC is the column that actually separates them.

In [ ]:
PRIMARY = "0.5"   # must match eval.primary_severity in the config

for clinic in ["montgomery", "shenzhen"]:
    m = json.loads(Path(f"{OUT}/baseline/{clinic}/metrics.json").read_text())
    print(f"=== {clinic} (split_mode={m['split_mode']}) ===")
    print(f"temperature={m['temperature']:.4f}  "
          f"val ECE {m['val_ece_before_temperature']:.4f} -> {m['val_ece_after_temperature']:.4f}")
    if m.get("temperature_at_bound"):
        print("  WARNING:", m["temperature_warning"])

    sweep = m["robustness_sweep"][PRIMARY]
    print(f"accuracy={sweep['accuracy']:.3f}  sens={sweep['sensitivity']:.3f}  "
          f"spec={sweep['specificity']:.3f}  ECE={sweep['ece']:.3f}  MCE={sweep['mce']:.3f}")

    for method, r in sweep["uncertainty_methods"].items():
        op, resc = r["operating_point"], r["human_rescue_rate"]
        print(f"  {method:<12} AURC={r['aurc']:.4f}  coverage={op['coverage']:.3f}  "
              f"acc_on_kept={op['accuracy']:.3f}  deferred={resc['deferred']}  "
              f"would_correct={resc['would_correct_frac']:.3f}")

    cp = m["conformal"]
    print(f"  conformal(alpha={cp['alpha']}): q_hat={cp['q_hat']:.3f} "
          f"attainable={cp['guarantee_attainable']} "
          f"in-dist cov={cp['in_distribution_reference']['coverage']:.3f} "
          f"heldout cov={cp['heldout_clinic']['coverage']:.3f} "
          f"shortfall={cp['coverage_shortfall_vs_target']:.3f}")
    print()

The conformal **shortfall** is the cross-site diagnostic: the guarantee is exact on data exchangeable with calibration, and however far the held-out clinic falls below it is a distribution-free readout of how far that site has drifted. Do not report it as a guarantee *on* the held-out clinic.

## 4. Accuracy across degradation severity

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5.5, 3.6))
for clinic in ["montgomery", "shenzhen"]:
    m = json.loads(Path(f"{OUT}/baseline/{clinic}/metrics.json").read_text())
    sev = sorted(m["robustness_sweep"], key=float)
    ax.plot([float(s) for s in sev],
            [m["robustness_sweep"][s]["accuracy"] for s in sev], "o-", label=clinic)
ax.axhline(REFERENCE_ACCURACY, ls="--", c="gray", label="in-distribution reference")
ax.set_xlabel("smartphone degradation severity")
ax.set_ylabel("accuracy on the held-out clinic")
ax.set_title("Robustness across capture quality")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## 5. Figures from the runs

In [ ]:
from IPython.display import Image, display

for clinic in ["montgomery", "shenzhen"]:
    p = Path(f"{OUT}/baseline/{clinic}/reliability_and_deferral.png")
    if p.exists():
        print(clinic)
        display(Image(str(p)))

gap = Path(f"{WORK}/loco_sweep/generalization_gap.png")
if gap.exists():
    display(Image(str(gap)))

## 6. Qualitative error analysis

Where are the missed TB cases concentrated? False negatives are the costly error in screening — a missed case seeds onward transmission.

In [ ]:
import pandas as pd

preds = pd.read_csv(f"{WORK}/loco_sweep/combined_predictions.csv")
preds["pred"] = (preds["prob"] >= 0.5).astype(int)
preds["correct"] = preds["pred"] == preds["label"]

summary = preds.groupby("clinic").apply(
    lambda g: pd.Series({
        "n": len(g),
        "accuracy": g["correct"].mean(),
        "false_negatives": int(((g["label"] == 1) & (g["pred"] == 0)).sum()),
        "false_positives": int(((g["label"] == 0) & (g["pred"] == 1)).sum()),
    }),
    include_groups=False,
)
print(summary)

Report per-fold `n` alongside every accuracy: Montgomery's fold is 138 images, so its point estimates carry wide intervals and should be shown with them.

Phase 5 (writing) pulls its tables and figures straight from these outputs — see `docs/PAPER_OUTLINE.md`.